In [11]:
import os
import shutil
import stat
from pathlib import Path

BASE_DIR = Path('../data/aptos')
target_splits = ['val_images', 'test_images']

def handle_remove_readonly(func, path, exc_info):
    """Clear Windows read-only flags so rmtree can delete directories."""
    os.chmod(path, stat.S_IWRITE)
    func(path)

for split in target_splits:
    split_dir = BASE_DIR / split
    if not split_dir.exists():
        print(f"Directory {split_dir} does not exist. Skipping...")
        continue

    # 1. Recursively find and move all nested PNGs to split_dir root
    all_pngs = list(split_dir.glob('**/*.png'))
    moved_count = 0
    for img in all_pngs:
        if img.parent != split_dir:
            target_path = split_dir / img.name
            shutil.move(str(img), str(target_path))
            moved_count += 1

    print(f"Moved {moved_count} images out of subfolders in {split}")

    # 2. Delete the empty nested subdirectories
    for item in split_dir.iterdir():
        if item.is_dir():
            try:
                shutil.rmtree(item, onexc=handle_remove_readonly)
            except Exception as e:
                print(f"Could not delete folder {item.name}: {e}")

print("\n--- Final Image Counts ---")
for split in ['train_images', 'val_images', 'test_images']:
    p = BASE_DIR / split
    if p.exists():
        count = len([f for f in os.listdir(p) if f.endswith('.png')])
        print(f"{split}: {count} PNGs")

Moved 0 images out of subfolders in val_images
Moved 0 images out of subfolders in test_images

--- Final Image Counts ---
train_images: 2930 PNGs
val_images: 366 PNGs
test_images: 366 PNGs
